In [14]:
import pandas as pd
import numpy as np

In [15]:
weighted_returns_df = pd.read_csv("data/outputs/baseline_portfolio",
                                  index_col = 0, parse_dates = [1])
daily_df_annually = pd.read_csv("data/outputs/annually_reblanced_portfolio",
                                index_col = 0, parse_dates = [1])
daily_df_quarterly = pd.read_csv("data/outputs/quarterly_reblanced_portfolio",
                                 index_col = 0, parse_dates = [1])
daily_df_quarterly_band = pd.read_csv("data/outputs/quarterly_reblanced_byband_portfolio", 
                                      index_col = 0, parse_dates = [1])
annual_weights_frame = pd.read_csv("data/outputs/annually_rebalanced_weights",
                                index_col = 0, parse_dates=[1])
quarterly_weights_frame= pd.read_csv("data/outputs/quarterly_rebalanced_weights",
                                index_col = 0)
band_weights_frame = pd.read_csv("data/outputs/quarterly_rebalanced_byband_weights",
                                index_col = 0)

In [17]:
## Portfolio Drift ##

def drift_evaluation(df):
    starting_weights = df[df["weight_type"] == "starting"]
    ending_weights = df[df["weight_type"] == "ending"]

    difference = np.array(ending_weights.drop(columns=['Date','weight_type'])) - np.array(starting_weights.drop(columns=['Date','weight_type']))

    drift_list = []

    for period in range(len(difference)):
        start_date = starting_weights['Date'].iloc[period]
        end_date = ending_weights['Date'].iloc[period]

        drift = abs(difference[period]).sum()
        drift_list.append({
            "Start": start_date,
            "End" : end_date,
            "Drift": drift
            })

    return pd.DataFrame(drift_list)

In [18]:
## Trades ##
def calculate_trades(df):
    starting_weights = df[df["weight_type"] == "starting"]
    ending_weights = df[df["weight_type"] == "ending"]
    starting_weights_next = starting_weights.drop(starting_weights.index[0])
    ending_weights_current = ending_weights.drop(ending_weights.index[-1])
    start = starting_weights_next.drop(columns=['Date','weight_type'])
    end = ending_weights_current.drop(columns=['Date','weight_type'])
    trades = np.array(start) - np.array(end)
    trades_df = pd.DataFrame(trades, index = starting_weights_next['Date'], columns = start.columns)
    return trades_df


In [19]:

annual_trades_df = calculate_trades(annual_weights_frame)
annual_trades_df

,AGG,EEM,EFA,GLD,IWM,LQD,MTUM,QQQ,QUAL,SPY,TLT,USMV,VLUE,VNQ
Date,,,,,,,,,,,,,,
2023-01-03,-0.011494,0.000000e+00,-8.960803e-03,0.150000,1.500000e-01,0.150000,0.000000e+00,-0.041728,0.000000e+00,-0.101925,-1.276777e-01,-0.081914,0.060631,-1.369304e-01
2024-01-02,0.010076,2.884593e-17,-6.754119e-03,0.000757,-1.547203e-01,0.005111,2.259489e-17,0.150000,1.500000e-01,-0.005700,5.181237e-16,-0.034013,-0.114756,4.465968e-17
2025-01-02,0.016766,1.500000e-01,-1.360986e-01,-0.016569,-8.730215e-17,0.017359,1.500000e-01,-0.115151,-1.608208e-01,0.095253,-4.176816e-16,-0.000739,0.000000,-4.103932e-17
2026-01-02,0.020659,-1.166472e-02,5.308254e-16,-0.047497,1.242124e-16,0.019810,-1.473898e-01,0.001424,1.257812e-16,0.007956,4.662246e-16,0.006702,0.150000,0.000000e+00


In [20]:
## Turnover ## 
def calculate_turnover(df):
    turnover_list = []
    for row in range(len(df.index)):
        turnover = abs(df.iloc[row]).sum(axis = 0) * 0.5
        turnover_list.append({
            'Trade Date':df.index[row],
            'turnover': turnover
            })
    return pd.DataFrame(turnover_list)


In [21]:

annual_portfolio_turnover = calculate_turnover(annual_trades_df)
annual_portfolio_turnover

,Trade Date,turnover
0,2023-01-03,0.510631
1,2024-01-02,0.315944
2,2025-01-02,0.429378
3,2026-01-02,0.206552


In [49]:
def value_analysis(df, turnover_df, bp):
    value_after_list = []
    transaction_cost = bp / 10000

    cost = 0

    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])

    trade_days = pd.to_datetime(turnover_df["Trade Date"])
    turnover = turnover_df["turnover"].tolist()

    for trade_day, turnover_value in zip(trade_days, turnover):
        prior_rows = df.loc[df["Date"] < trade_day]

        prior_rows['portfolio_value'] = prior_rows['portfolio_value'] - cost

        prior_row = prior_rows.iloc[-1]
        portfolio_value = prior_row["portfolio_value"]
        cost = portfolio_value * turnover_value * transaction_cost

        value_after = portfolio_value - cost

        value_after_list.append({
            "valuation_day" : prior_row['Date'],
            "value": portfolio_value,
            "reblance_day": trade_day,
            "cost": cost,
            "rebalance_day" : trade_day,
            "portfolio_value": value_after
        })

    return pd.DataFrame(value_after_list)

value_analysis(
    df = daily_df_annually,
    turnover_df=annual_portfolio_turnover,
    bp = 10
)

,valuation_day,value,reblance_day,cost,rebalance_day,portfolio_value
0,2022-12-30,0.811157,2023-01-03,0.000414,2023-01-03,0.810743
1,2023-12-29,0.921762,2024-01-02,0.000291,2024-01-02,0.921471
2,2024-12-31,1.050534,2025-01-02,0.000451,2025-01-02,1.050083
3,2025-12-31,1.302630,2026-01-02,0.000269,2026-01-02,1.302361


In [47]:

annually_rebalanced_tc = value_analysis(
    df = daily_df_annually,
    turnover_df=annual_portfolio_turnover,
    bp = 10
)
annually_rebalanced_tc


(  valuation_day     value reblance_day      cost rebalance_day  \
 0    2022-12-30  0.811157   2023-01-03  0.000414    2023-01-03   
 1    2023-12-29  0.921762   2024-01-02  0.000291    2024-01-02   
 2    2024-12-31  1.050534   2025-01-02  0.000451    2025-01-02   
 3    2025-12-31  1.302630   2026-01-02  0.000269    2026-01-02   
 
    portfolio_value  
 0         0.810743  
 1         0.921471  
 2         1.050083  
 3         1.302361  ,
            Date  portfolio_return   rf_rate  portfolio_value  running_peak  \
 0    2022-01-03         -0.003996  0.000003         0.995553      0.996004   
 1    2022-01-04         -0.000310  0.000003         0.995245      0.996004   
 2    2022-01-05         -0.013461  0.000004         0.981842      0.996004   
 3    2022-01-06         -0.001078  0.000004         0.980783      0.996004   
 4    2022-01-07         -0.003593  0.000004         0.977258      0.996004   
 ...         ...               ...       ...              ...           ...   

In [ ]:
## Value After Transaction ##



## Weight Stability ## 
For each rebalance:

Compute turnover
T
Observe portfolio value before the rebalance
V
Assume transaction cost
c
Cost paid
Cost=V×T×c
New portfolio value
V
after
	​

=V−Cost

Holding Period Analysis

Questions

Average holding period

How often is each ETF traded?

Average position age

Number of consecutive quarters held